In [3]:
from PIL import Image
import os
import math

def create_image_grid(folder_path, output_path, grid_size=None, image_size=(200, 200)):
    """
    Combine all images in a folder into a grid image, sorted by filename.

    Args:
        folder_path (str): path to folder containing images
        output_path (str): path to save output image
        grid_size (tuple[int, int]): (cols, rows). If None, auto-calculate as square grid
        image_size (tuple[int, int]): resize each image to this size before combining
    """
    # Lấy danh sách ảnh trong folder
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print("⚠️ Không tìm thấy ảnh nào trong thư mục.")
        return

    # 🔧 SẮP XẾP ảnh theo tên file (thứ tự thời gian)
    image_files.sort()
    print(f"📁 Tìm thấy {len(image_files)} ảnh")
    print(f"📋 Thứ tự sắp xếp:")
    for i, f in enumerate(image_files, 1):
        print(f"   {i}. {f}")

    images = [Image.open(os.path.join(folder_path, f)).resize(image_size) for f in image_files]
    num_images = len(images)

    # Nếu chưa chỉ định grid_size thì chọn dạng vuông gần nhất
    if grid_size is None:
        cols = int(math.ceil(math.sqrt(num_images)))
        rows = int(math.ceil(num_images / cols))
    else:
        cols, rows = grid_size

    # Kích thước ảnh đầu ra
    grid_width = cols * image_size[0]
    grid_height = rows * image_size[1]

    # Tạo ảnh trắng để dán
    grid_image = Image.new('RGB', (grid_width, grid_height), color=(255, 255, 255))

    # Dán từng ảnh vào lưới
    for idx, img in enumerate(images):
        x = (idx % cols) * image_size[0]
        y = (idx // cols) * image_size[1]
        grid_image.paste(img, (x, y))

    # Lưu kết quả
    grid_image.save(output_path)
    print(f"\n✅ Đã ghép {num_images} ảnh thành công!")
    print(f"📊 Grid: {cols} cột × {rows} hàng")
    print(f"💾 Output: {output_path}")


# --- Ví dụ sử dụng ---
if __name__ == "__main__":
    create_image_grid(
        folder_path="/home/serverai/ltdoanh/LayoutGeneration/checkcheck/run_tv2_dists_v11_filtered_v11/keyframes",       # thư mục chứa ảnh
        output_path="grid_output.jpg", # ảnh kết quả
        image_size=(256, 256)          # kích thước mỗi ảnh
    )

📁 Tìm thấy 16 ảnh
📋 Thứ tự sắp xếp:
   1. scene_0000_frame_00000000.jpg
   2. scene_0001_frame_00000070.jpg
   3. scene_0002_frame_00000140.jpg
   4. scene_0003_frame_00000200.jpg
   5. scene_0004_frame_00000252.jpg
   6. scene_0005_frame_00000316.jpg
   7. scene_0006_frame_00000356.jpg
   8. scene_0007_frame_00000472.jpg
   9. scene_0008_frame_00000545.jpg
   10. scene_0009_frame_00000618.jpg
   11. scene_0010_frame_00000698.jpg
   12. scene_0011_frame_00000757.jpg
   13. scene_0012_frame_00000836.jpg
   14. scene_0013_frame_00000873.jpg
   15. scene_0014_frame_00000999.jpg
   16. scene_0015_frame_00001034.jpg

✅ Đã ghép 16 ảnh thành công!
📊 Grid: 4 cột × 4 hàng
💾 Output: grid_output.jpg


In [5]:
import os
import json
import pandas as pd

# Path đến thư mục chứa các kết quả
base_dir = "/home/serverai/ltdoanh/LayoutGeneration/outputs/Dang/eval/eval/lpips"

# Danh sách để lưu kết quả
results = []

# Loop qua các folder con trong outputs_eval
for subdir, dirs, files in os.walk(base_dir):
    if "eval_results.json" in files:
        json_path = os.path.join(subdir, "eval_results.json")
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
                data["folder"] = os.path.basename(subdir)  # thêm tên folder để theo dõi
                results.append(data)
        except Exception as e:
            print(f"Error reading {json_path}: {e}")

# Chuyển sang DataFrame để dễ xử lý
df = pd.DataFrame(results)

# Tính trung bình các metric (bỏ cột folder)
summary = df.drop(columns=["folder"]).mean().to_frame(name="Mean").T

print("=== Summary of Metrics Across All Evaluations ===")
print(summary)

# Xuất ra file CSV nếu muốn
summary.to_csv("summary_eval_metrics.csv", index=False)


=== Summary of Metrics Across All Evaluations ===
        RecErr   Frechet  SceneCoverage  TemporalCoverage@tau  \
Mean  0.407009  0.838171            1.0              0.245414   

      RedundancyMeanCos  MinPairwiseDist  Sharpness_med  Exposure_med  \
Mean           0.453211         0.312022     284.753039    175.646996   

      Noise_med    NumKeys  NumAllEmbed  
Mean   5.836062  40.377778   144.355556  
